# 14장 · 분류 함수와 조건부 라우팅

라우팅 함수는 입력의 분류 결과에 따라 다음에 실행할 함수를 선택합니다. 분류가 모호하거나 실패했을 때 갈 경로도 정해야 합니다.

영수증 문의와 인증 오류를 서로 다른 처리 함수로 보냅니다. 분류 함수의 반환값, 파이썬 조건문이 선택한 경로, 추가 확인이 필요한 입력을 비교합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## 분류 레이블과 복합 의도

실제 모델이 반환한 분류로 처리기를 선택합니다. 모호한 질문과 API 호출 실패를 다른 이유로 기록하세요.

### 명시적 거절과 분류기 오류

<p>route_request는 ClassificationRefusalError를 만나면 classification_refusal을, 그 밖의 예외를 만나면 classification_error를 반환합니다. 두 경우 모두 작성기를 호출하지 않습니다. 셀의 timed_out은 TimeoutError를 일으켜 두 번째 경로를 보여 줍니다.</p>


### 셀 01 · 실제 모델 호출

키워드 영수증이 있고 오류는 없는 입력입니다.


In [ ]:
import os
from openai import OpenAI

client = OpenAI(timeout=45, max_retries=1)
model_name = os.getenv("OPENAI_MODEL") or "gpt-4o-mini"
def classify_question(question):
    rule = "영수증 등 결제 문의는 billing, 기술 오류는 technical, 두 의도가 섞이거나 모호하면 clarify. 라벨 하나만 출력하세요."
    return client.responses.create(model=model_name, instructions=rule, input=question).output_text.strip()
print(classify_question("영수증이 필요합니다"))

billing


billing이 나옵니다. 두 단어가 모두 있거나 모두 없으면 이 함수는 분류를 확정하지 않고 clarify를 반환합니다.

‘영수증 오류가 납니다’에는 두 신호가 모두 들어 있습니다. 이 입력을 어느 한 팀으로 보내면 다른 요청을 놓칠 수 있습니다. 조건문에서 두 신호가 같은지 먼저 검사하는 이유를 두 단어가 모두 없는 문의와도 비교해 보세요.


### 분류 함수와 실행 함수

<p>분류 모델은 입력이 어느 분야인지 문자열을 제안합니다. 그 문자열로 어떤 작업을 수행할지는 Python 코드가 결정합니다. 모델이 billing이라고 썼다는 이유만으로 결제나 환불이 실제 실행된 것은 아닙니다.</p>

<p>이번 예제는 접수 안내 문장만 만듭니다. 외부 계정 조회·환불·데이터 변경은 하지 않습니다. 경로 분류는 사용자 권한이나 작업 승인도 대신하지 않습니다.</p>

<table><caption>순차 흐름과 분기</caption><thead><tr><th scope="col">구분</th><th scope="col">실행 방식</th><th scope="col">확인점</th></tr>
</thead><tbody><tr><td>13장 순차</td><td>요약→분류→작성 고정</td><td>중간 출력 전달</td></tr>
<tr><td>14장 분기</td><td>분류→결제 또는 기술 작성</td><td>선택된 규칙만 실행</td></tr>
<tr><td>추가 질문</td><td>분류 후 고정 질문 반환</td><td>작성 모델 호출 없음</td></tr>
</tbody></table>


### 과제 02 · 복합 의도의 clarify 분기

영수증과 오류가 모두 있는 한 문장을 만들어 분류값을 확인하세요.


In [ ]:
# 복합 문의 하나를 classify_question에 넣으세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>print(classify_question(&quot;영수증 출력 중 오류가 났어요&quot;))</code></pre>
<p>실행 결과</p><pre><code>clarify</code></pre>
<p>두 조건이 모두 참이라 결과는 clarify입니다. 어떤 분야인지 더 물은 뒤에 작성할 수 있습니다.</p>
</details>


#### 두 의도가 함께 들어온 요청

<mark>분류 레이블 하나가 곧 실행할 작업을 결정합니다.</mark> 결제와 다른 문의가 섞였을 때 먼저 걸린 조건만 따라가면 나머지 의도를 잃을 수 있습니다. clarify가 반환된 경우에는 분류 근거를 확인하고 무엇을 되물을지 정하세요.

선수 개념 · 13장 순차 워크플로의 입력·출력 연결

#### 분류기의 시간 초과는 어느 경로로 돌아올까요?

```mermaid
flowchart TD
nin["질문"]
  nclassify["분류 호출"]
  nwrite["작성"]
  nexcept["예외 처리"]
  nreturn["결과"]
  nin --> nclassify
  nclassify --> nwrite
  nclassify --> nexcept
  nexcept --> nreturn
  nwrite --> nreturn
```

timed_out을 분류 함수로 넣으면 결제 단서가 있는 질문도 작성 함수까지 도달하지 못합니다. 여기서 clarify의 원인은 classification_error입니다.

### 입력 유형별 라우팅 테스트

<table><caption>분류 예시</caption><thead><tr><th scope="col">질문</th><th scope="col">경로</th><th scope="col">작성 횟수</th></tr>
</thead><tbody><tr><td>영수증이 필요합니다</td><td>billing</td><td>1</td></tr>
<tr><td>인증 오류입니다</td><td>technical</td><td>1</td></tr>
<tr><td>영수증과 오류</td><td>clarify</td><td>0</td></tr>
<tr><td>도와주세요</td><td>clarify</td><td>0</td></tr>
</tbody></table>

<p>분류 모델은 문의 문맥을 읽어 billing, technical, clarify 중 하나를 반환합니다. 실제 LLM의 의미 이해를 흉내 낸 결과로 평가하지 않습니다. “결제 내역을 받고 싶어요”처럼 같은 의도를 다른 표현으로 쓰면 이 규칙은 추가 질문으로 보냅니다.</p>


## route_request()의 분기와 실패 처리

허용하는 작성 규칙은 billing과 technical 둘입니다. 규칙을 찾을 수 있으면 한 번 쓰고, 그렇지 않으면 추가 설명을 요청합니다. write는 선택된 분기의 지시문과 원문을 실제 모델에 전달합니다.

<!-- lesson-visual practice-space -->
**영수증과 오류가 동시에 나오면 어디로 갈까요?**

수업의 분류 함수는 두 단어의 포함 여부를 검사합니다. 질문을 바꾸면서 강조되는 경로와 안내 작성 함수의 호출 수를 확인하세요.

두 단서 중 하나만 있을 때는 해당 경로에서 1회 작성합니다. 둘 다 있거나 모두 없으면 clarify로 이동하고 작성 횟수는 0입니다.

[웹 교재에서 직접 조작하기](https://langchain-lecture.vercel.app/chapter14/#practice-space)
<!-- /lesson-visual practice-space -->


### 순차 실행에서 조건부 실행으로

<p>13장의 작성 단계 앞에 조건이 생겼습니다. technical이면 작성기를 부르고, clarify면 추가 질문을 돌려줍니다.</p>


### 셀 03 · route_request()로 실행 경로 선택

제공 함수의 writes는 최종 안내 작성 함수를 실제로 몇 번 불렀는지 표시합니다.

분류기가 빈 문자열을 그대로 반환하면 작성 규칙을 찾지 못해 needs_clarification으로 끝납니다. SDK 연결부가 먼저 빈 응답을 예외로 처리하면 route_request는 classification_error를 기록합니다. 어느 쪽도 작성 함수를 부르지 않습니다.


In [ ]:
WRITING_RULES = {"billing": "결제 문의의 접수 안내를 작성하세요.", "technical": "기술 문의의 확인 항목을 작성하세요."}

class ClassificationRefusalError(ValueError):
    """명시적 거절을 정상적인 추가 질문과 구분하는 예외입니다."""

def route_request(classify, write, question):
    if not question.strip():
        raise ValueError("빈 질문")
    try:
        route = classify(question)
    except ClassificationRefusalError as exc:
        return {"route": "clarify", "reason": "classification_refusal",
                "error_type": type(exc).__name__, "answer": "분류 요청이 거절되었습니다. 요청 내용을 확인하세요.", "writes": 0}
    except Exception as exc:
        return {"route": "clarify", "reason": "classification_error",
                "error_type": type(exc).__name__, "answer": "분류에 실패했습니다. 잠시 뒤 다시 확인하세요.", "writes": 0}
    if route not in WRITING_RULES:
        return {"route": "clarify", "reason": "needs_clarification",
                "answer": "결제 문의인지 기술 문제인지 구체적으로 알려 주세요.", "writes": 0}
    answer = write(WRITING_RULES[route], question)
    return {"route": route, "reason": "classified", "answer": answer, "writes": 1}

def write_reply(instruction, question):
    return client.responses.create(model=model_name, instructions=instruction, input=question).output_text

result = route_request(classify_question, write_reply, "인증 오류입니다")
print(result["route"], result["writes"])

technical 1


technical 1은 기술 문의 규칙으로 작성 함수를 한 번 실행했다는 뜻입니다. 분류 호출은 writes에 들어가지 않습니다.


### 허용 경로 검증

<p>WRITING_RULES의 키가 허용 목록입니다. 분류 결과가 딕셔너리에 없으면 코드가 추가 질문 경로를 선택합니다. 모델이 반환한 문자열을 eval로 실행하거나 같은 이름의 임의 함수를 찾지 않습니다.</p>

<table><caption>분류 이후의 처리</caption><thead><tr><th scope="col">분류 호출 결과</th><th scope="col">reason</th><th scope="col">작성 호출</th></tr></thead><tbody><tr><td>billing 또는 technical</td><td>classified</td><td>1회</td></tr><tr><td>clarify 또는 미등록 문자열</td><td>needs_clarification</td><td>0회</td></tr><tr><td>ClassificationRefusalError</td><td>classification_refusal</td><td>0회</td></tr><tr><td>TimeoutError</td><td>classification_error</td><td>0회</td></tr></tbody></table>

### 셀 04 · 분류 시간 초과 처리

분류기가 답을 돌려주기 전에 시간 초과가 났다고 가정합니다.


In [ ]:
def timed_out(question):
    raise TimeoutError("분류 대기 시간 초과")

failed = route_request(timed_out, write_reply, "영수증이 필요합니다")
print(failed["reason"], failed["writes"])

classification_error 0


classification_error 0입니다. 문의가 모호했기 때문이 아니라 분류 과정에서 실패해 작성까지 가지 못했습니다.


### 분류 예외와 정상적인 추가 질문

<p>분류 함수가 TimeoutError를 내면 reason은 classification_error입니다. 두 분야의 단서가 함께 있어 분류 함수가 clarify를 돌려주면 needs_clarification입니다. 두 경우 모두 writes는 0이지만 원인 기록이 다릅니다.</p>


## clarify 경로와 추가 질문

복합 문의와 시스템 오류의 화면 문구가 같으면 사용자가 올바른 다음 행동을 고르기 어렵습니다. 한 건의 분류 기록에서 이유와 안내를 함께 읽습니다.


### 경계 입력과 원인별 로그

<p>분류는 쉬운 질문보다 모호·복합·범위 밖 입력에서 설계 차이가 드러납니다. 두 분야를 동시에 묻는 질문에 하나만 골라 답하면 나머지 요청이 사라질 수 있습니다. 이번 수업에서는 복합 입력을 추가 질문으로 돌려 범위를 좁힙니다.</p>

<p>등록되지 않은 경로가 반환되면 route not in WRITING_RULES가 참입니다. clarify를 받은 경우와 같은 추가 질문을 돌려주며 write는 호출하지 않습니다.</p>

### 과제 05 · clarify 분기의 응답값

‘도와주세요’를 route_request에 넣고 기록에서 사용자에게 보여 줄 답변을 꺼내세요.


In [5]:
# 요청을 처리한 뒤 answer를 출력하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>ambiguous = route_request(classify_question, write_reply, &quot;도와주세요&quot;)
print(ambiguous[&quot;answer&quot;])</code></pre>
<p>실행 결과</p><pre><code>결제 문의인지 기술 문제인지 구체적으로 알려 주세요.</code></pre>
<p>결제 문의인지 기술 문제인지 구체적으로 알려 달라는 안내입니다. 분류 실패의 ‘잠시 뒤 다시 확인’과는 다음 행동이 다릅니다.</p>
</details>


### 분기 정책 비교 실습

<p>이 장의 route는 분기 규칙을 찾을 때 쓰는 문자열입니다. 다음 장에서는 함수 이름뿐 아니라 인자와 호출 식별자까지 포함한 요청을 읽습니다.</p>

## 복합 문의를 되묻고 선택한 경로로 이어 보내기

“영수증 출력 중 오류가 나요”에는 두 가지 단서가 있습니다. 분류기가 추가 질문을 냈을 때 안내문을 먼저 작성하지 않고, 사용자가 결제와 기술 중 하나를 고른 뒤에만 해당 작성 함수를 부릅니다.

### 셀 08 · 겹치는 두 단서를 한 요청으로 보내기

기존 route_request에 복합 문의를 보냅니다. 사용자에게 보낼 질문과 실제 작성 호출 횟수를 확인하세요.

In [ ]:
mixed_question = "영수증 출력 중 오류가 나요."
waiting = route_request(classify_question, write_reply, mixed_question)
print(waiting["reason"], waiting["writes"])
print(waiting["answer"])

classified 1
안녕하세요!

영수증 출력 중 오류가 발생한 점 사과드립니다. 문제를 해결하기 위해 다음 단계를 확인해주세요:

1. **프린터 연결 상태**: 프린터가 제대로 연결되어 있는지 확인하세요.
2. **프린터 드라이버**: 드라이버가 최신 버전인지 확인하고, 필요시 업데이트 해주세요.
3. **종이 및 잉크 확인**: 프린터에 종이가 충분히 있거나 잉크가 부족하지 않은지 확인해주세요.

위 사항을 모두 점검한 후에도 문제가 해결되지 않는다면, 고객센터로 문의해 주시면 신속히 도와드리겠습니다.

감사합니다!


needs_clarification과 0이 나옵니다. 복합 문의가 어느 쪽인지 아직 정하지 않았으므로 결제 안내와 기술 안내 모두 작성하지 않은 상태입니다.

### 과제 09 · 사용자 선택으로 원래 문의 이어가기

사용자가 “기술 문제”를 눌렀다고 가정하세요. 이 선택을 technical로 해석한 뒤 기존 route_request에 원래 문의를 다시 넘기세요. 작성 함수는 받은 지시문과 문의를 기록해야 합니다. 실제로 선택한 경로와 작성 함수가 받은 문의를 출력하세요.

In [7]:
selection = "기술 문제"
choice_codes = {"결제 문의": "billing", "기술 문제": "technical"}
# 분류 함수는 selection에 해당하는 경로를 돌려주고 작성 함수는 받은 문의를 기록하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>selection = &quot;기술 문제&quot;
choice_codes = {&quot;결제 문의&quot;: &quot;billing&quot;, &quot;기술 문제&quot;: &quot;technical&quot;}
write_inputs = []
def use_selection(question):
    return choice_codes.get(selection, &quot;clarify&quot;)

def record_write(instruction, question):
    write_inputs.append((instruction, question))
    return write_reply(instruction, question)

resumed = route_request(use_selection, record_write, mixed_question)
print(resumed[&quot;route&quot;], len(write_inputs))
print(write_inputs[0][1])</code></pre>
<p>실행 결과</p><pre><code>technical 1
영수증 출력 중 오류가 나요.</code></pre>
<p>technical에서 안내를 한 번 작성했고 작성 함수에는 처음의 복합 문의가 그대로 넘어갔습니다. 선택 문구만 작성자에게 넘기면 영수증 출력 중이었다는 상황이 빠집니다.</p>
</details>

<mark>경로를 고르는 답과 실제로 처리할 문의 원문을 따로 보관하세요.</mark> “기술 문제”는 담당 경로를 가리킵니다. 어떤 오류가 났는지 설명하는 입력은 첫 번째 문의에 남아 있습니다.

도구 실행 단계에서도 실행할 도구의 이름과 그 도구에 전달할 인자를 따로 확인합니다.

## 돌아보기

정상 분류, 복합 문의, 분류 시간 초과 중 한 건을 골라 어디서 분기하는지 설명해 보세요. reason으로 분기 이유를, writes로 작성 함수를 몇 번 불렀는지 확인할 수 있습니다.